This is the career alter ego project

1. it ingests my CV and professional summary
2. it evaluates the response sent to it

then the imports

In [53]:

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
from pydantic import BaseModel
import os
import requests
import json

load the env variables

In [54]:
load_dotenv(override=True)
openai = OpenAI()

now, lets load in my CV and career summary and define my name

In [55]:
reader = PdfReader("me/cv.pdf")
cv_data = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        cv_data += text

career_summary = ""
with open("me/summary.txt", "r", encoding="utf-8") as f:
    career_summary = f.read()

name = "Adeyemi Adetayo"

Next, lets setup an evaluator, if this chat bot will be acting as me, the response must be good

In [56]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [64]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and CV details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{career_summary}\n\n## cv:\n{cv_data}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

lets setup pusher and pusher tool to notify us when we dont have answer to the question

In [58]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    response = requests.post(pushover_url, data=payload)
    print(response.json())

def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

record_unknown_question_json = {
    "type": "function",
    "function": {
        "name": "record_unknown_question",
        "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "The question that couldn't be answered"
                },
            },
            "required": ["question"],
            "additionalProperties": False
        }
    }
}

lets test pusher

In [59]:
push("Hello, world!")

Push: Hello, world!
{'status': 1, 'request': 'a6a5f236-0ce5-4ebe-8bcc-cd60ef886bd6'}


Lets define our tool handler

In [60]:

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

Lets setup system prompt for our chat model

In [61]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and CV which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career."

system_prompt += f"\n\n## Summary:\n{career_summary}\n\n## CV:\n{cv_data}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

system_prompt

"You are acting as Adeyemi Adetayo. You are answering questions on Adeyemi Adetayo's website, particularly questions related to Adeyemi Adetayo's career, background, skills and experience. Your responsibility is to represent Adeyemi Adetayo for interactions on the website as faithfully as possible. You are given a summary of Adeyemi Adetayo's background and CV which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career.\n\n## Summary:\nAdeyemi Adetayo is a product-focused software engineering leader and fintech innovator with over five years of experience building and scaling high-impact digital systems. With a foundation in Petroleum Engineering and a career pivot into software development, he brings a u

Lets setup the rerun function for our chat model

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages,tools=[record_unknown_question_json])
    return response

In [69]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages,tools=[record_unknown_question_json])
    while response.choices[0].message.tool_calls:
        messages.append(response.choices[0].message)
        tool_calls = response.choices[0].message.tool_calls
        messages += handle_tool_calls(tool_calls)
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages,tools=[record_unknown_question_json])
        reply = response.choices[0].message.content if response.choices[0].message.content else "/n".join([tool_call.function.name for tool_call in response.choices[0].message.tool_calls])
        print("reply to evaluate:",reply)
        evaluation = evaluate(reply, message, history)
        print("evaluation:",evaluation)
        if not evaluation.is_acceptable:
            print("Failed evaluation - retrying")
            response = rerun(reply, message, history, evaluation.feedback)
    return response.choices[0].message.content
        

Lets bring it all together

In [70]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


Tool called: record_unknown_question
Push: Recording Where exactly did Adeyemi Adetayo grow up? asked that I couldn't answer
{'status': 1, 'request': '52f9c196-51d7-4d2a-bbb5-28e15ab9d304'}
reply to evaluate: I'm sorry, but I don't have information on where I grew up. If you have any questions about my career, skills, or experiences, feel free to ask!
evaluation: is_acceptable=True feedback="The agent correctly identified that the requested information (where Adeyemi grew up) is not available in the provided context. It then professionally redirected the user to ask about relevant topics that are covered in the persona's information (career, skills, experiences), maintaining an engaging and helpful tone consistent with the instructions."
